# Gaussian kernel or learned features

This notebook accompanies Exercise 5.4. The data form two intertwined crescents: an affine score is too rigid, but two non-affine constructions become possible.

The Gaussian SVM fixes a kernel and then solves a convex problem. The MLP learns its representation together with its final classifier. We compare their boundaries, errors and sensitivity to hyperparameters.

## Outline

1. [Two crescents and a deterministic split](#demi-lunes)
2. [Affine SVM and Gaussian SVM](#svm-noyau)
3. [MLP and learned features](#mlp-lunes)
4. [Boundaries and complexity](#comparaison-lunes)
5. [Deterministic perturbations](#perturbations-lunes)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import flax
from flax import nnx
import sklearn
from sklearn.svm import SVC

print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

<a id="demi-lunes"></a>
## 1. Two crescents and a deterministic split

For $\theta_i=i\pi/(m-1)$, construct
$$
x_i^+=(\cos\theta_i,\sin\theta_i),\qquad
x_i^-=(1-\cos\theta_i,1/2-\sin\theta_i).
$$

Use indices modulo $5$ to obtain a reproducible split: three indices for training, one for validation and one for testing. Apply the same rule to both classes.

In [ ]:
# Complete this section.

In [ ]:
# Complete this section.

<a id="svm-noyau"></a>
## 2. Affine SVM and Gaussian SVM

We use the `SVC` class from `scikit-learn`, whose implementation is based on LIBSVM.

1. Fit an affine SVM with `SVC(kernel="linear")`.
2. For the Gaussian kernel
   $$K_\gamma(x,x')=\exp(-\gamma\lVert x-x'\rVert^2),$$
   choose $C$ and $\gamma$ by minimising the validation error frequency.
3. Never use the test data to make this choice.
4. Plot the support vectors of the selected Gaussian model.

In [ ]:
# Complete this section.

<a id="mlp-lunes"></a>
## 3. MLP and learned features

The following class and training function are provided. Fit several widths and initialisations. Select the machine using the validation data, as for the kernel.

Here, the penalty acts on all MLP parameters.

In [ ]:
class MLPBinaire(nnx.Module):
    def __init__(self, dimension, largeur, *, rngs):
        self.affine_1 = nnx.Linear(dimension, largeur, rngs=rngs)
        self.affine_2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.affine_3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        y = jnp.tanh(self.affine_1(x))
        y = jnp.tanh(self.affine_2(y))
        return self.affine_3(y)[..., 0]

def norme_parametres(machine):
    return sum(
        jnp.sum(feuille**2)
        for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param))
    )

def cout_mlp(machine, x, z, lamb):
    return (
        jnp.mean(jax.nn.softplus(-z * machine(x)))
        + 0.5 * lamb * norme_parametres(machine)
    )

@nnx.jit
def pas_mlp(machine, optimiseur, x, z, lamb):
    fonction = lambda m: cout_mlp(m, x, z, lamb)
    valeur, gradient = nnx.value_and_grad(fonction)(machine)
    optimiseur.update(machine, gradient)
    return valeur

def entrainer_mlp(x, z, largeur, graine, *, lamb=1.0e-4, iterations=1500):
    machine = MLPBinaire(x.shape[1], largeur, rngs=nnx.Rngs(graine))
    optimiseur = nnx.Optimizer(
        machine, optax.adam(1.0e-2), wrt=nnx.Param
    )
    x_jax = jnp.asarray(x)
    z_jax = jnp.asarray(z)
    for _ in range(iterations):
        valeur = pas_mlp(machine, optimiseur, x_jax, z_jax, lamb)
    return machine, float(valeur)

In [ ]:
# Complete this section.

<a id="comparaison-lunes"></a>
## 4. Boundaries and complexity

Plot all three boundaries on the same domain. Compare:

- training, validation and test error frequencies;
- the number of support vectors of the Gaussian SVM;
- the number of MLP parameters;
- each construction's sensitivity to its hyperparameters.

The number of support vectors and number of parameters are not directly comparable measures of effective complexity.

In [ ]:
g1 = np.linspace(-1.4, 2.4, 320)
g2 = np.linspace(-1.0, 1.4, 260)
G1, G2 = np.meshgrid(g1, g2)
grille = np.column_stack([G1.ravel(), G2.ravel()])

def tracer_modele(axe, score, titre, supports=None):
    valeurs = np.asarray(score(grille)).reshape(G1.shape)
    axe.contourf(G1, G2, valeurs, levels=[-100, 0, 100], colors=["C0", "C3"], alpha=0.15)
    axe.contour(G1, G2, valeurs, levels=[0.0], colors="k")
    for etiquette, couleur, marqueur in [(-1, "C0", "o"), (1, "C3", "s")]:
        masque = z_app == etiquette
        axe.scatter(x_app[masque, 0], x_app[masque, 1], c=couleur, marker=marqueur, s=18)
    if supports is not None:
        axe.scatter(
            supports[:, 0], supports[:, 1],
            s=65, facecolors="none", edgecolors="k", label="support vectors"
        )
        axe.legend(loc="lower right", fontsize=8)
    axe.set_title(titre)
    axe.set(xlabel=r"$x_1$", ylabel=r"$x_2$")
    axe.grid(True)

In [ ]:
# Complete this section.

In [ ]:
# Complete this section.

<a id="perturbations-lunes"></a>
## 5. Deterministic perturbations

Add deterministic perturbations of increasing amplitude to both crescents, for example combinations of $\sin(7\theta)$ and $\cos(11\theta)$. Repeat the comparison for a few amplitudes.

Does the most flexible boundary always give the lowest test error frequency? Why would selection using only training data be misleading?

In [ ]:
# Complete this section.

## Summary

A kernel SVM and an MLP both construct non-affine boundaries, but learn different objects. The former chooses a classifier in a fixed feature space; the latter also learns this space. Comparing their results requires the same split and selection rule.